In [10]:
from dotenv import load_dotenv
import os
import tweepy

# 1. Baca file `.env` sehingga variabel BEARER_TOKEN terdefinisi
load_dotenv()  
BEARER_TOKEN = os.getenv("BEARER_TOKEN")

# 2. Inisialisasi tweepy.Client dengan Bearer Token
#    (tidak memerlukan API Key / Secret Key / Access Token / Secret untuk read-only search v2)
client = tweepy.Client(
    bearer_token=BEARER_TOKEN,
    wait_on_rate_limit=True
)

# kredensial Twitter
API_KEY = "cenQPC6sHXBvRZRCSnAXRGpSn"
API_SECRET_KEY = "5DPhzgsdyjoYm0FJQljlxYmcHySaBPbVrUMsFpZuFv0TSvZTQU"
ACCESS_TOKEN = "1930678349162270720-d3jCmmf0eJcKIrLmbtdTJsMeUPprJo"
ACCESS_TOKEN_SECRET = "tN4TDom4m0LYBa2JltytmWEXSog2o7x7HNKnN05IsNI6Q"

# Cell 3: Autentikasi OAuth dan inisialisasi objek "api"
auth = tweepy.OAuth1UserHandler(API_KEY, API_SECRET_KEY, ACCESS_TOKEN, ACCESS_TOKEN_SECRET)
api = tweepy.API(auth, wait_on_rate_limit=True)



In [12]:
def search_recent_tweets_v2(query, max_results=10, lang="id"):
    """
    Mencari tweet publik terbaru menggunakan endpoint API v2 (recent search).

    Args:
        query (str): Kata kunci atau hashtag yang ingin dicari (contoh: "#python").
                     Anda bisa menambahkan filter khusus di query, misal `#python lang:en`.
        max_results (int): Jumlah maksimum tweet yang diambil (1–100 per panggilan, tergantung level akses).
        lang (str): Kode bahasa ISO-639-1 (contoh: "id" untuk Bahasa Indonesia, "en" untuk Inggris).
                    Jika ingin filter bahasa, tambahkan `lang:xx` di dalam query (contoh: `f"{query} lang:en"`).

    Returns:
        List of tweepy.Tweet: Daftar objek Tweet dari API v2.
    """
    # Jika ingin filter bahasa di query v2, formatnya: "kata_kunci lang:ID"
    # Misal: query="MBG lang:id"
    full_query = f"{query} lang:{lang}" if lang else query

    # Panggil endpoint recent_search
    response = client.search_recent_tweets(
        query=full_query,
        max_results=max_results,
        tweet_fields=["created_at", "author_id", "lang", "public_metrics"], 
        # Anda bisa menambahkan fields lain: "text", dll.
    )

    # `response.data` berisi list objek Tweet; jika tidak ada hasil, bisa None
    tweets = response.data if response.data is not None else []
    return tweets


In [14]:
# Contoh: Cari 5 tweet terbaru yang mengandung "MBG" berbahasa Indonesia
kata_kunci = "MBG"
jumlah = 100

print(f"Mencari {jumlah} tweet terbaru dengan kata kunci '{kata_kunci}' (bahasa ID)...\n")
hasil = search_recent_tweets_v2(kata_kunci, max_results=jumlah, lang="id")

# Cetak hasil
for idx, tweet in enumerate(hasil, start=1):
    # tweet.id, tweet.text, tweet.created_at, tweet.author_id, tweet.public_metrics tersedia
    teks = tweet.text.replace("\n", " ")  # memadatkan baris baru jika ada
    waktu = tweet.created_at
    print(f"{idx}. [waktu: {waktu}] ID Tweet: {tweet.id}")
    print(f"    {teks}\n")


Mencari 100 tweet terbaru dengan kata kunci 'MBG' (bahasa ID)...



Rate limit exceeded. Sleeping for 869 seconds.


1. [waktu: 2025-06-05 22:19:43+00:00] ID Tweet: 1930751416295121031
    @dowoonique mbg mengkudeta bang gibran

2. [waktu: 2025-06-05 22:19:42+00:00] ID Tweet: 1930751410003382421
    RT @nababastala: ini pemerintahan kerja gk sih? PHK massal, daya beli turun, abuse of power dimana-mana, SDA dieksploitasi, utang meroket,…

3. [waktu: 2025-06-05 22:19:28+00:00] ID Tweet: 1930751350180389138
    RT @nababastala: ini pemerintahan kerja gk sih? PHK massal, daya beli turun, abuse of power dimana-mana, SDA dieksploitasi, utang meroket,…

4. [waktu: 2025-06-05 22:19:04+00:00] ID Tweet: 1930751249399623685
    RT @Bofurin_Fess: 🍃🐮Selamat hari raya Idul Adha🐮🍃  Yang seharusnya dikorbankan : sapi, kambing, dan unta.   Bukan  - hutan di Papua - anak-…

5. [waktu: 2025-06-05 22:18:54+00:00] ID Tweet: 1930751209440518315
    RT @nababastala: ini pemerintahan kerja gk sih? PHK massal, daya beli turun, abuse of power dimana-mana, SDA dieksploitasi, utang meroket,…

6. [waktu: 2025-06-05 22:18:54+00:0

In [ ]:
import pandas as pd

# Asumsikan `hasil` adalah list objek Tweet dari API v2 yang sudah diperoleh sebelumnya.
# Contoh setiap objek Tweet memiliki atribut: id, text, created_at, author_id, public_metrics

# 1. Bangun list of dict dari setiap tweet:
data = []
for tweet in hasil:
    data.append({
        'id': tweet.id,
        'text': tweet.text,
        'created_at': tweet.created_at,
        'author_id': tweet.author_id,
        'retweet_count': tweet.public_metrics['retweet_count'],
        'reply_count': tweet.public_metrics['reply_count'],
        'like_count': tweet.public_metrics['like_count'],
        'quote_count': tweet.public_metrics['quote_count']
    })

# 2. Buat DataFrame
df = pd.DataFrame(data)

# 3. Simpan ke file CSV
csv_path = "/mnt/data/tweets.csv"
df.to_csv(csv_path, index=False)

# 4. Tampilkan DataFrame dalam notebook
import ace_tools as tools; tools.display_dataframe_to_user(name="Hasil Scraping Tweet", dataframe=df)

# 5. Informasi lokasi file CSV untuk diunduh
print(f"CSV tersimpan di: {csv_path}")
